# Entra Agent ID setup (Lab1)

Step-by-step notebook: **blueprint + principal**, **client secret**, **agent identity**.

## Prerequisites

- Entra tenant with Agent ID enabled; permission to create agent blueprints (role or tenant setting).
- Python 3.10+ and Jupyter (VS Code / Cursor).
- Install dependencies: `pip install -r requirements.txt` (in this folder).

## Security

**Do not commit this notebook with outputs saved** — Step 2 shows a client secret once.

## Graph API version

This lab uses **Microsoft Graph v1.0** endpoints. If a call fails with v1.0, see [Create blueprint](https://learn.microsoft.com/en-us/entra/agent-id/create-blueprint) and try the equivalent beta URL only as a fallback.

**Out of scope here:** permissions, token exchange, API test — use [`EntraAgentID-Functions.ps1`](EntraAgentID-Functions.ps1) for those steps.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import random
import time
from datetime import datetime, timedelta, timezone

import httpx
import msal

TENANT_ID = "00000000-0000-0000-0000-000000000000"  # @@@ YOUR TENANT ID @@@
RUN_SUFFIX = random.randint(100000, 999999)
BLUEPRINT_NAME = f"Lab1 Agent Blueprint {RUN_SUFFIX}"
AGENT_NAME = f"Lab1 Weather Agent {RUN_SUFFIX}"
print(f"Run suffix: {RUN_SUFFIX}")
print(f"Blueprint name: {BLUEPRINT_NAME}")
print(f"Agent name: {AGENT_NAME}")

GRAPH_ROOT = "https://graph.microsoft.com/v1.0"
# Public client for device-code sign-in (Microsoft's Graph PowerShell app, same as Connect-MgGraph).
# Replace with your own app registration client_id if your tenant requires it.
DELEGATED_AUTH_CLIENT_ID = "14d82eec-204b-4c2f-b7e8-296a70dab67e"

DELEGATED_SCOPES = [
    "https://graph.microsoft.com/AgentIdentityBlueprint.Create",
    "https://graph.microsoft.com/AgentIdentityBlueprint.AddRemoveCreds.All",
    "https://graph.microsoft.com/AgentIdentityBlueprintPrincipal.Create",
    "https://graph.microsoft.com/User.Read",
]

if "00000000" in TENANT_ID:
    raise ValueError("Set TENANT_ID in the configuration cell before continuing.")

In [ ]:
def acquire_delegated_token(tenant_id: str, scopes: list[str], client_id: str) -> str:
    """Device code flow for admin Graph calls (steps 1–2)."""
    authority = f"https://login.microsoftonline.com/{tenant_id}"
    app = msal.PublicClientApplication(client_id, authority=authority)
    flow = app.initiate_device_flow(scopes=scopes)
    if "user_code" not in flow:
        raise RuntimeError(f"Device flow failed: {flow}")
    print(flow["message"])
    result = app.acquire_token_by_device_flow(flow)
    if "access_token" not in result:
        raise RuntimeError(f"Sign-in failed: {result.get('error_description', result)}")
    return result["access_token"]


def graph_request(
    method: str,
    url: str,
    token: str,
    json_body: dict | None = None,
    extra_headers: dict | None = None,
) -> httpx.Response:
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "OData-Version": "4.0",
    }
    if extra_headers:
        headers.update(extra_headers)
    with httpx.Client(timeout=60.0) as client:
        response = client.request(method, url, headers=headers, json=json_body)
    if response.status_code >= 400:
        raise RuntimeError(f"{method} {url} -> {response.status_code}: {response.text}")
    return response


def acquire_blueprint_token(
    tenant_id: str,
    app_id: str,
    client_secret: str,
    retries: int = 5,
    delay_seconds: float = 3.0,
) -> str:
    """Client credentials for blueprint (step 3)."""
    authority = f"https://login.microsoftonline.com/{tenant_id}"
    app = msal.ConfidentialClientApplication(
        app_id,
        authority=authority,
        client_credential=client_secret,
    )
    last_error = None
    for attempt in range(1, retries + 1):
        result = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
        if "access_token" in result:
            return result["access_token"]
        last_error = result.get("error_description") or result
        if attempt < retries:
            print(f"Token not ready (attempt {attempt}/{retries}), waiting {delay_seconds}s...")
            time.sleep(delay_seconds)
    raise RuntimeError(f"Failed to acquire blueprint token: {last_error}")

## Admin sign-in (device code)

Uses `DELEGATED_AUTH_CLIENT_ID` for the device code flow (default: Microsoft Graph PowerShell app). Some tenants require your own app registration with the same delegated permissions and admin consent.

In [ ]:
admin_token = acquire_delegated_token(TENANT_ID, DELEGATED_SCOPES, DELEGATED_AUTH_CLIENT_ID)
me = graph_request("GET", f"{GRAPH_ROOT}/me", admin_token).json()
user_id = me["id"]
print(f"Signed in as: {me.get('userPrincipalName', me.get('displayName'))}")
print(f"User object id (sponsor): {user_id}")

## Step 1 — Blueprint + blueprint principal

Creates the agent identity blueprint and its blueprint principal in the tenant.

In [ ]:
blueprint_body = {
    "displayName": BLUEPRINT_NAME,
    "sponsors@odata.bind": [f"{GRAPH_ROOT}/users/{user_id}"],
    "owners@odata.bind": [f"{GRAPH_ROOT}/users/{user_id}"],
}

blueprint_url = f"{GRAPH_ROOT}/applications/microsoft.graph.agentIdentityBlueprint"
blueprint_resp = graph_request("POST", blueprint_url, admin_token, blueprint_body).json()

blueprint_app_id = blueprint_resp["appId"]
blueprint_object_id = blueprint_resp["id"]
print(f"Blueprint appId (client id): {blueprint_app_id}")
print(f"Blueprint object id: {blueprint_object_id}")

principal_body = {"appId": blueprint_app_id}
principal_url = f"{GRAPH_ROOT}/servicePrincipals/microsoft.graph.agentIdentityBlueprintPrincipal"
principal_resp = graph_request("POST", principal_url, admin_token, principal_body).json()
principal_id = principal_resp["id"]
print(f"Blueprint principal id: {principal_id}")

print("Waiting 5s for principal propagation...")
time.sleep(5)

## Step 2 — Client secret

Adds a password credential to the blueprint. **Copy `CLIENT_SECRET` now** — it is not shown again.

In [ ]:
secret_end = (datetime.now(timezone.utc) + timedelta(days=365)).strftime("%Y-%m-%dT%H:%M:%SZ")
password_body = {
    "passwordCredential": {
        "displayName": f"Agent ID Secret {BLUEPRINT_NAME}",
        "endDateTime": secret_end,
    }
}

add_password_url = f"{GRAPH_ROOT}/applications/{blueprint_object_id}/addPassword"
secret_resp = graph_request("POST", add_password_url, admin_token, password_body).json()
CLIENT_SECRET = secret_resp.get("secretText")
if not CLIENT_SECRET:
    raise RuntimeError(f"addPassword did not return secretText: {secret_resp}")

print("CLIENT_SECRET saved to variable (length):", len(CLIENT_SECRET))
print("\n--- CLIENT SECRET (copy now) ---\n")
print(CLIENT_SECRET)
print("\n--- end secret ---")

## Step 3 — Agent identity

Uses blueprint **client credentials** (not the admin token) to create the agent.

In [ ]:
blueprint_token = acquire_blueprint_token(TENANT_ID, blueprint_app_id, CLIENT_SECRET)

agent_body = {
    "displayName": AGENT_NAME,
    "agentIdentityBlueprintId": blueprint_app_id,
    "sponsors@odata.bind": [f"{GRAPH_ROOT}/users/{user_id}"],
}
agent_url = f"{GRAPH_ROOT}/servicePrincipals/microsoft.graph.agentIdentity"
agent_resp = graph_request("POST", agent_url, blueprint_token, agent_body).json()

agent_identity_app_id = agent_resp["appId"]
agent_identity_sp_id = agent_resp["id"]
print(f"Agent display name: {agent_resp.get('displayName', AGENT_NAME)}")
print(f"Agent appId: {agent_identity_app_id}")
print(f"Agent service principal id: {agent_identity_sp_id}")

## Summary

Use `lab_context` with other Lab1 scripts. **Do not commit outputs that contain `CLIENT_SECRET`.**

In [ ]:
lab_context = {
    "tenant_id": TENANT_ID,
    "run_suffix": RUN_SUFFIX,
    "blueprint_name": BLUEPRINT_NAME,
    "blueprint_app_id": blueprint_app_id,
    "blueprint_object_id": blueprint_object_id,
    "principal_id": principal_id,
    "user_id": user_id,
    "client_secret": CLIENT_SECRET,
    "agent_name": AGENT_NAME,
    "agent_identity_app_id": agent_identity_app_id,
    "agent_identity_sp_id": agent_identity_sp_id,
}

display_keys = {k: v for k, v in lab_context.items() if k != "client_secret"}
print("lab_context (secret omitted from print):")
for key, value in display_keys.items():
    print(f"  {key}: {value}")

print("\nNext steps (PowerShell): permissions, token exchange, weather API setup — see EntraAgentID-Functions.ps1 and setup-weather-api-resource.ps1")